In [1]:
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path().resolve().parents[0]
sys.path.append(str(ROOT))
CHROMA_DB_PATH = str(ROOT / "chroma_db")

In [2]:
import pandas as pd
import os
import ast
import openai
import time
import pickle
#from utils.get_openai_embedding import get_openai_embedding
from utils.neo4j_utils import run_query

In [3]:
from utils.neo4j_utils import driver, run_query

try:
    driver.verify_connectivity()  # lightweight handshake
    result = run_query("RETURN 'driver connected' AS status")
    print(result[0]["status"])
except Exception as err:
    print(f"Neo4j connection failed: {err}")


driver connected


In [ ]:
# Dataset after preprocessing, weight calculation and tagging

df = pd.read_pickle('../data/pp_recipes_preprocessed_v3_weighted_and_tagged_1000_profiled.pkl') # reaplace with pp_recipes_preprocessed_v3  

In [ ]:
df.iloc[5]

In [5]:
df.columns

Index(['title', 'total_energy_kcal_per_serving_hummus',
       'total_energy_from_fat_kcal_per_serving_hummus',
       'total_fat_g_per_serving_hummus',
       'total_saturated_fat_g_per_serving_hummus',
       'total_cholesterol_mg_per_serving_hummus',
       'total_sodium_mg_per_serving_hummus',
       'total_carbohydrate_g_per_serving_hummus',
       'total_dietary_fiber_g_per_serving_hummus',
       'total_sugar_g_per_serving_hummus',
       'total_protein_g_per_serving_hummus', 'who_score_hummus',
       'fsa_score_hummus', 'nutri_score_hummus',
       'total_protein_g_per_serving_irish',
       'total_carbohydrate_g_per_serving_irish',
       'total_fat_g_per_serving_irish', 'total_energy_kcal_per_serving_irish',
       'total_sugar_g_per_serving_irish',
       'total_saturated_fat_g_per_serving_irish',
       'total_sodium_mg_per_serving_irish', 'duration', 'directions',
       'servingsPerRecipe', 'servingSize [g]', 'calories [cal]',
       'caloriesFromFat [cal]', 'totalFat [g

In [ ]:
df.dtypes

In [ ]:
df.iloc[0]['ingredient_names'][0]

# Create ingredient and tag embeddings

In [6]:
# Unique ingredients

def get_unique_ingredients(df, column='ingredient_names'):
    unique_ingredients = set()
    for ingredients in df[column]:
        unique_ingredients.update(ingredients)
    return sorted(unique_ingredients)

unique_ingredients = get_unique_ingredients(df)

In [7]:
len(unique_ingredients)

4067

In [8]:
# Unique tags

def get_unique_tags(df, column="tags"):
    unique_tags = set()
    for tags in df[column]:
        unique_tags.update(tags)
    return sorted(unique_tags)

unique_tags = get_unique_tags(df)
print(unique_tags)


['- Dinner', '- Lunch', '15-minutes-or-less', '30-minutes-or-less', '4-hours-or-less', '5-ingredients-or-less', '60-minutes-or-less', 'African', 'Almond', 'Almonds', 'Amaretto', 'American', 'Anchovies', 'Anise-flavored liqueur', 'Appetizer', 'Apple', 'Apples', 'Applesauce', 'Appleton Rum', 'Apricots', 'Artichoke', 'Artichokes', 'Asian', 'Asparagus', 'Australian', 'Avocado', 'Avocados', 'Bacon', 'Baileys', 'Baking', 'Baking Soda', 'Baking mix', 'Balsamic vinegar', 'Banana', 'Banana Peppers', 'Bananas', 'Barbecued Beef', 'Barley', 'Basil', 'Bean Sprouts', 'Beans', 'Beef', 'Beef Liver', 'Beef Suet', 'Beer', 'Beetroots', 'Belgian', 'Bell Pepper', 'Bell peppers', 'Beninese', 'Berries', 'Beverages and Cocktails', 'Biscuit', 'Biscuits', 'Bisquick', 'Bitter Lemon', 'Black Beans', 'Black beans', 'Blackberry', 'Blackberry Brandy', 'Blanching', 'Blending', 'Blue Cheese', 'Blue Curacao', 'Blueberries', 'Blueberry Vodka', 'Blueberry vodka', 'Boiling', 'Bologna', 'Bourbon', 'Brandy', 'Bratwursts', '

In [9]:
len(unique_tags)

552

In [ ]:
unique_tags

In [10]:
import sys, os

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

if "tools" in sys.modules:
    del sys.modules["tools"]

from tools.ingredient_embeddings_tool import ensure_ingredients_in_collection

result = ensure_ingredients_in_collection.invoke({
    "ingredient_names": unique_ingredients,
    "state": {
        "persist_path": CHROMA_DB_PATH,      # optional override
        "collection_name": "ingredients", # optional override
        "debug": True                     # turn on progress prints if you want
    },
})
print(result)

/home/karvanitis/RecipeWrangler-Backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[ensure_ingredients_in_collection] persist_path=/home/karvanitis/RecipeWrangler-Backend/chroma_db collection=ingredients
[input] 4067 names
[existing] found=4067 missing=0
[done] found=4067 created=0 failed=0 total=13418
{'persist_path': '/home/karvanitis/RecipeWrangler-Backend/chroma_db', 'collection': 'ingredients', 'found': ['16 total', '4 slices bread, of your choice or 2 English muffins, sliced', '9-inch baked pie crust', 'Absolut citron vodka', 'Accent seasoning', 'Agave', 'Agave (to taste)', 'Amaretto', 'Asian chili sauce', 'Baby Spinach', 'Baby Spinach, chopped', 'Baby Spinach, shredded', 'Baileys Irish Cream', 'Bisquick', 'Bisquick baking mix', 'Bisquick reduced-fat baking mix', 'Bourbon', 'Brussels sprout', 'Brussels sprout, trimmed and halved', 'Brussels sprout, trimmed, halved and very thinly sliced', 'Brussels sprouts, finely sliced', 'Bulgar wheat', 'Carnation Evaporated Milk', 'Catalina dressing (original)', 'Cavenders All Purpose Greek Seasoning, divided in half', 'Cham

In [11]:
import sys, os

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

if "tools" in sys.modules:
    del sys.modules["tools"]

from tools.tags_embeddings_tool import ensure_tags_in_collection

result = ensure_tags_in_collection.invoke({
    "tag_names": unique_tags,
    "state": {
        "persist_path": CHROMA_DB_PATH,  # optional override
        "collection_name": "tags",    # optional override
        "debug": True                 # turn on progress prints if you want
    },
})
print(result)

[ensure_tags_in_collection] persist_path=/home/karvanitis/RecipeWrangler-Backend/chroma_db collection=tags
[input] 552 names
[existing] found=552 missing=0
[done] found=552 created=0 failed=0 total=2298
{'persist_path': '/home/karvanitis/RecipeWrangler-Backend/chroma_db', 'collection': 'tags', 'found': ['- Dinner', '- Lunch', '15-minutes-or-less', '30-minutes-or-less', '4-hours-or-less', '5-ingredients-or-less', '60-minutes-or-less', 'African', 'Almond', 'Almonds', 'Amaretto', 'American', 'Anchovies', 'Anise-flavored liqueur', 'Appetizer', 'Apple', 'Apples', 'Applesauce', 'Appleton Rum', 'Apricots', 'Artichoke', 'Artichokes', 'Asian', 'Asparagus', 'Australian', 'Avocado', 'Avocados', 'Bacon', 'Baileys', 'Baking', 'Baking Soda', 'Baking mix', 'Balsamic vinegar', 'Banana', 'Banana Peppers', 'Bananas', 'Barbecued Beef', 'Barley', 'Basil', 'Bean Sprouts', 'Beans', 'Beef', 'Beef Liver', 'Beef Suet', 'Beer', 'Beetroots', 'Belgian', 'Bell Pepper', 'Bell peppers', 'Beninese', 'Berries', 'Bever

# Delete nodes , relationships and indexes from previous sessions

In [12]:
def delete_all_data_and_indexes():
    queries = [
        "MATCH (n)-[r]->() DELETE r",
        "MATCH (n) DETACH DELETE n",
        
        "DROP INDEX ingredient_vector_index IF EXISTS",
        "DROP INDEX tag_vector_index IF EXISTS"
    ]
    for query in queries:
        run_query(query)

delete_all_data_and_indexes()

# Create Nodes and Relationships

In [13]:
import json
import math
import numpy as np
import pandas as pd
import numbers
from collections.abc import Mapping, Sequence

LIST_KEYS = {
    "directions",
    "ingredient_names",
    "measurements",
    "tags",
    "allergens",
    "weights",
}
ALLOWED_SCALARS = (str, int, float, bool)

def sanitize_for_neo4j(value):
    """Collapse pandas/numpy types and nested structures into Neo4j-safe values."""
    if value is None or value is pd.NA:
        return None

    if isinstance(value, (np.generic,)):
        value = value.item()

    if isinstance(value, float) and math.isnan(value):
        return None

    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()

    if isinstance(value, Mapping):
        sanitized = {}
        for k, v in value.items():
            cleaned = sanitize_for_neo4j(v)
            if cleaned is None:
                continue
            sanitized[str(k)] = cleaned
        return json.dumps(sanitized)

    if isinstance(value, Sequence) and not isinstance(value, (str, bytes)):
        cleaned = []
        for item in value:
            item_clean = sanitize_for_neo4j(item)
            if item_clean is None:
                continue
            if isinstance(item_clean, list):
                item_clean = json.dumps(item_clean)
            cleaned.append(item_clean)

        if not cleaned:
            return None

        element_types = {type(x) for x in cleaned}
        if len(element_types) > 1:
            if all(isinstance(x, numbers.Real) and not isinstance(x, bool) for x in cleaned):
                cleaned = [float(x) for x in cleaned]
            else:
                cleaned = [str(x) for x in cleaned]

        return cleaned

    if isinstance(value, ALLOWED_SCALARS):
        return value

    return str(value)

def validate_row(row):
    """Optional debug helper: reports the first Neo4j-incompatible value."""
    for key, raw in row.items():
        sanitized = sanitize_for_neo4j(raw)
        if sanitized is None:
            continue
        if isinstance(sanitized, list) and not all(isinstance(v, ALLOWED_SCALARS) for v in sanitized):
            return key, sanitized
        if not isinstance(sanitized, (list, *ALLOWED_SCALARS)):
            return key, sanitized
    return None

# Optional one-time sanity check
for idx, row in df.iterrows():
    bad = validate_row(row)
    if bad:
        print(f"Row {idx} has Neo4j-incompatible value at `{bad[0]}`: {bad[1]!r}")
        break
else:
    print("All rows sanitize cleanly.")


# change hummus to irish for composition table
def create_recipe_nodes(df):
    query = """
    MERGE (r:Recipe {title: $title})
    SET r.duration = $duration,
        r.instructions = $directions,
        r.serves = $servingsPerRecipe,
        r.servingsizegrams = $serving_size_g,
        r.totalenergykcalperservinghummus = $total_energy_kcal_per_serving_hummus,
        r.totalenergyfromfatkcalperserving = $total_energy_from_fat_kcal_per_serving_hummus,
        r.totalfatgperserving = $total_fat_g_per_serving_hummus,
        r.totalsaturatedfatgperserving = $total_saturated_fat_g_per_serving_hummus,
        r.totalcholesterolmgperserving = $total_cholesterol_mg_per_serving_hummus,
        r.totalsodiummgperservinghummus = $total_sodium_mg_per_serving_hummus,
        r.totalcarbohydrategperserving = $total_carbohydrate_g_per_serving_hummus,
        r.totaldietaryfibergperserving = $total_dietary_fiber_g_per_serving_hummus,
        r.totalsugargperserving = $total_sugar_g_per_serving_hummus,
        r.totalproteingperserving = $total_protein_g_per_serving_hummus,
        r.directionsize = $direction_size,
        r.ingredientssizes = $ingredients_sizes,
        r.whoscore = $who_score_hummus,
        r.fsascore = $fsa_score_hummus,
        r.nutriscore = $nutri_score_hummus,
        r.totalsustainability = $total_sustainability,
        r.totalsustainabilityperserving = $total_sustainability_per_serving,
        r.sustainabilityperkg = $sustainability_per_kg
    RETURN r
    """


    for _, row in df.iterrows():
        raw_params = {
            "title": row["title"],
            "duration": row["duration"],
            "directions": row["directions"],
            "servingsPerRecipe": row["servingsPerRecipe"],
            "serving_size_g": row["servingSize [g]"],
            "total_energy_kcal_per_serving_hummus": row["total_energy_kcal_per_serving_hummus"],
            "total_energy_from_fat_kcal_per_serving_hummus": row["total_energy_from_fat_kcal_per_serving_hummus"],
            "total_fat_g_per_serving_hummus": row["total_fat_g_per_serving_hummus"],
            "total_saturated_fat_g_per_serving_hummus": row["total_saturated_fat_g_per_serving_hummus"],
            "total_cholesterol_mg_per_serving_hummus": row["total_cholesterol_mg_per_serving_hummus"],
            "total_sodium_mg_per_serving_hummus": row["total_sodium_mg_per_serving_hummus"],
            "total_carbohydrate_g_per_serving_hummus": row["total_carbohydrate_g_per_serving_hummus"],
            "total_dietary_fiber_g_per_serving_hummus": row["total_dietary_fiber_g_per_serving_hummus"],
            "total_sugar_g_per_serving_hummus": row["total_sugar_g_per_serving_hummus"],
            "total_protein_g_per_serving_hummus": row["total_protein_g_per_serving_hummus"],
            "direction_size": row["direction_size"],
            "ingredients_sizes": row["ingredients_sizes"],
            "who_score_hummus": row["who_score_hummus"],
            "fsa_score_hummus": row["fsa_score_hummus"],
            "nutri_score_hummus": row["nutri_score_hummus"],
            "ingredient_names": row["ingredient_names"],
            "measurements": row["measurements"],
            "tags": row["tags"],
            "allergens": row["allergens"],
            "weights": row["weights"],
            "total_protein_g_per_serving_irish": row["total_protein_g_per_serving_irish"],
            "total_carbohydrate_g_per_serving_irish": row["total_carbohydrate_g_per_serving_irish"],
            "total_fat_g_per_serving_irish": row["total_fat_g_per_serving_irish"],
            "total_energy_kcal_per_serving_irish": row["total_energy_kcal_per_serving_irish"],
            "total_sugar_g_per_serving_irish": row["total_sugar_g_per_serving_irish"],
            "total_saturated_fat_g_per_serving_irish": row["total_saturated_fat_g_per_serving_irish"],
            "total_sodium_mg_per_serving_irish": row["total_sodium_mg_per_serving_irish"],
            "total_sustainability": row["total_sustainability"],
            "total_sustainability_per_serving": row["total_sustainability_per_serving"],
            "sustainability_per_kg": row["sustainability_per_kg"],
        }

        params = {}
        for key, value in raw_params.items():
            clean = sanitize_for_neo4j(value)
            if clean is None and key in LIST_KEYS:
                clean = []               # Neo4j still expects the parameter for list fields
            params[key] = clean

        run_query(query, params)

create_recipe_nodes(df)


All rows sanitize cleanly.


In [14]:
def create_ingredient_nodes(df):
    query = """
    MERGE (i:Ingredient {name: $ingredient})
    WITH i
    MATCH (r:Recipe {title: $title})
    MERGE (r)-[rel:HAS_INGREDIENT]->(i)
    SET rel.measurement = $measurement,
        rel.weight = $weight
    """

    for _, row in df.iterrows():
        ingredients = row["ingredient_names"] or []
        measurements = row["measurements"] or []
        weights = row["weights"] or []

        for ingredient, measurement, weight in zip(ingredients, measurements, weights):
            run_query(
                query,
                {
                    "title": row["title"],
                    "ingredient": ingredient,
                    "measurement": measurement,
                    "weight": weight,
                },
            )

create_ingredient_nodes(df)

In [15]:
# Add ingredients embeddings

def add_ingredient_embeddings(ingredient_embeddings):
    query = """
    MATCH (i:Ingredient {name: $name})
    SET i.embedding = $embedding
    """
    for name, embedding in ingredient_embeddings.items():
        run_query(query, parameters={"name": name, "embedding": embedding})
        # print(f"Stored embedding for: {name}")

import chromadb

PERSIST_PATH = CHROMA_DB_PATH

def load_ingredient_embeddings(persist_path=PERSIST_PATH, collection_name="ingredients", batch_size=500):
    client = chromadb.PersistentClient(path=persist_path)
    collection = client.get_collection(collection_name)

    total = collection.count()
    embeddings = {}

    for offset in range(0, total, batch_size):
        batch = collection.get(
            include=["documents", "embeddings"],
            limit=batch_size,
            offset=offset,
        )
        names = batch.get("documents", [])
        vectors = batch.get("embeddings", [])

        for name, embedding in zip(names, vectors):
            if not name or embedding is None:
                continue
            if hasattr(embedding, "tolist"):
                embedding = embedding.tolist()
            embeddings[name] = embedding

    return embeddings

ingredient_embeddings = load_ingredient_embeddings()
add_ingredient_embeddings(ingredient_embeddings)


In [16]:
# Create the vector index for ingredients embeddings

def create_ingredient_vector_index():
    query = """
    CREATE VECTOR INDEX ingredient_vector_index
    FOR (i:Ingredient)
    ON (i.embedding)
    OPTIONS { indexConfig: {
      `vector.dimensions`: 4096,
      `vector.similarity_function`: 'cosine'
    }};
    """
    run_query(query)
    #print("Vector index 'ingredient_vector_index' created successfully.")

create_ingredient_vector_index()

In [17]:
# Create SIMILAR_TO relationships between ingredients

def create_similar_relationships(threshold=0.8, top_k=20):
    query = f"""
    MATCH (a:Ingredient)
    WHERE a.embedding IS NOT NULL
    CALL db.index.vector.queryNodes('ingredient_vector_index', {top_k}, a.embedding)
    YIELD node AS b, score
    WHERE a <> b AND id(a) < id(b) AND score > {threshold}
    MERGE (a)-[r:SIMILAR_TO]-(b)
    ON CREATE SET r.score = score
    RETURN a.name AS source, b.name AS target, score
    """
    
    results = run_query(query)
    #for record in results:
        #print(f"{record['source']} <--> {record['target']} (score: {record['score']:.3f})")

create_similar_relationships()

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated function: `id`.} {position: line: 6, column: 22, offset: 188} for query: "\n    MATCH (a:Ingredient)\n    WHERE a.embedding IS NOT NULL\n    CALL db.index.vector.queryNodes('ingredient_vector_index', 20, a.embedding)\n    YIELD node AS b, score\n    WHERE a <> b AND id(a) < id(b) AND score > 0.8\n    MERGE (a)-[r:SIMILAR_TO]-(b)\n    ON CREATE SET r.score = score\n    RETURN a.name AS source, b.name AS target, score\n    "
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated function: `id`.} {position: line: 6

In [18]:
def create_tag_nodes(df):
    merge_tag = """
    MERGE (t:Tag {name: $tag})
    """
    link_recipe = """
    MATCH (r:Recipe {title: $title})
    MATCH (t:Tag {name: $tag})
    MERGE (r)-[:HAS_TAG]->(t)
    """

    for _, row in df.iterrows():
        tags = row.get("tags") or []
        for tag in tags:
            if not tag:
                continue
            run_query(merge_tag, {"tag": tag})
            run_query(link_recipe, {"title": row["title"], "tag": tag})

create_tag_nodes(df)

In [19]:
# Add tag embeddings

def add_tag_embeddings(tag_embeddings):
    query = """
    MATCH (t:Tag {name: $name})
    SET t.embedding = $embedding
    """
    for name, embedding in tag_embeddings.items():
        run_query(query, parameters={"name": name, "embedding": embedding})
        # print(f"Stored embedding for tag: {name}")

import chromadb

PERSIST_PATH = CHROMA_DB_PATH

def load_tag_embeddings(persist_path=PERSIST_PATH, collection_name="tags", batch_size=500):
    client = chromadb.PersistentClient(path=persist_path)
    collection = client.get_collection(collection_name)

    total = collection.count()
    embeddings = {}

    for offset in range(0, total, batch_size):
        batch = collection.get(
            include=["documents", "embeddings"],
            limit=batch_size,
            offset=offset,
        )
        names = batch.get("documents", [])
        vectors = batch.get("embeddings", [])

        for name, embedding in zip(names, vectors):
            if not name or embedding is None:
                continue
            if hasattr(embedding, "tolist"):
                embedding = embedding.tolist()
            embeddings[name] = embedding

    return embeddings

tag_embeddings = load_tag_embeddings()
add_tag_embeddings(tag_embeddings)


In [20]:
# Create the vector index for tag embeddings

def create_tag_vector_index():
    query = """
    CREATE VECTOR INDEX tag_vector_index
    FOR (t:Tag)
    ON (t.embedding)
    OPTIONS { indexConfig: {
      `vector.dimensions`: 4096,
      `vector.similarity_function`: 'cosine'
    }};
    """
    run_query(query)
    #print("Vector index 'tag_vector_index' created successfully.")

create_tag_vector_index()


In [21]:
def create_allergen_nodes(df):
    merge_allergen = """
    MERGE (a:Allergen {name: $allergen})
    """
    link_recipe = """
    MATCH (r:Recipe {title: $title})
    MATCH (a:Allergen {name: $allergen})
    MERGE (r)-[:HAS_ALLERGEN]->(a)
    """

    for _, row in df.iterrows():
        allergens = row.get("allergens") or []
        for allergen in allergens:
            if not allergen:
                continue
            run_query(merge_allergen, {"allergen": allergen})
            run_query(link_recipe, {"title": row["title"], "allergen": allergen})

create_allergen_nodes(df)


# Helpful Functions

In [ ]:
# Function to search for recipes based on the desired ingredients

def find_similar_recipe_query(query, threshold=0.8, top_k=10):

    query_embedding = get_openai_embedding(query)

    cypher_query = f"""
    CALL db.index.vector.queryNodes('ingredient_vector_index', {top_k}, {query_embedding})
    YIELD node AS ing, score
    WHERE score > {threshold}

    MATCH (r:Recipe)-[:HAS_INGREDIENT]->(ing)
    RETURN DISTINCT r.title AS recipe, ing.name AS matched_ingredient, score
    ORDER BY score DESC
    LIMIT 20
    """

    results = run_query(cypher_query)

    if results:
        print("Recipes containing ingredients similar to your query:")
        for result in results:
            print(f"Recipe: {result['recipe']} (via ingredient: {result['matched_ingredient']}, score: {result['score']:.2f})")
    else:
        print("No recipes found with similar ingredients.")


In [ ]:
find_similar_recipe_query("chicken, onion")